# 合同审查助手 · Windows 进度功能重建版（Mac）

这是根据 **2026-11-16 今天已完成的功能和测试记录** 重建的最小可运行 Notebook，不是 Windows 本地 `notebooks/contract.ipynb` 的逐字拷贝。

**包含：** 3 条规则 WQ-001/002/003；`ContractReviewResult` Pydantic 输出结构；按原顺序读取 DOCX 段落与表格；逐条调用 SiliconFlow/DeepSeek 结构化审查；基本验收。

**不覆盖旧文件：** 本文件为 `notebooks/contract_review_win_rebuild.ipynb`；规则另存为 `data/rules/contract_rules_win_rebuild.json`，不修改 Mac 上原有的 `contract_rules.json`。

请使用仓库原有的 `wastewater-rag` Conda 环境。在 **项目根目录** 开启 Jupyter / VS Code。需要已有依赖：`python-docx`、`langchain-openai`、`python-dotenv`、`pydantic`。调用模型时仍使用你本机 `.env` 中已有的 `SILICONFLOW_API_KEY`，不要把密钥提交到 Git。

In [9]:
from pathlib import Path
import json

# 从仓库根目录或 notebooks 目录启动 Jupyter 均可。
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "src").exists(), "找不到项目根目录，请确认 Jupyter 工作目录"

RULES_PATH = PROJECT_ROOT / "data" / "rules" / "contract_rules_win_rebuild.json"
with RULES_PATH.open("r", encoding="utf-8") as f:
    rules = json.load(f)

assert len(rules) == 3
assert {r["rule_id"] for r in rules} == {"WQ-001", "WQ-002", "WQ-003"}
print("规则加载成功：", [(r["rule_id"], r["title"]) for r in rules])

规则加载成功： [('WQ-001', '水质超标处理与价格协商机制'), ('WQ-002', '水量计量与结算机制'), ('WQ-003', '水质采样与检测机制')]


## 1. 解析 DOCX（保留段落和表格出现的顺序）

这里使用 `doc.iter_inner_content()`，不会出现先抽取全文段落、再把所有表格放到末尾导致原文顺序错乱的问题。测试合同为**自动生成的虚构样例**；如果本地已经有同名文件，不覆盖。

In [31]:
from docx import Document
from docx.text.paragraph import Paragraph
from docx.table import Table


def load_contract_docx(file_path: str | Path) -> str:
    doc = Document(str(file_path))
    contract_blocks = []

    for block in doc.iter_inner_content():
        if isinstance(block, Paragraph):
            text = block.text.strip()
            if text:
                contract_blocks.append(text)
        elif isinstance(block, Table):
            table_rows = []
            for row in block.rows:
                cells = [cell.text.strip() for cell in row.cells]
                table_rows.append(" | ".join(cells))
            contract_blocks.append("[合同表格]\n" + "\n".join(table_rows))

    return "\n\n".join(contract_blocks)


TEST_DOCX = PROJECT_ROOT / "data" / "raw" / "contracts" / "test_contract_win_rebuild.docx"
TEST_DOCX.parent.mkdir(parents=True, exist_ok=True)
if not TEST_DOCX.exists():
    doc = Document()
    doc.add_heading("污水处理服务协议", level=1)
    doc.add_heading("6.3 水质超标处理费", level=2)
    doc.add_paragraph("甲方交付的污水水质超过本协议约定的接收标准时，乙方有权按照原污水处理单价的1.5倍收取超标处理费用。")
    doc.add_heading("6.4 水量结算", level=2)
    doc.add_paragraph("污水处理费用按照实际接收水量进行结算，甲方应按月支付污水处理费用。")
    doc.add_heading("5.1 水质检测", level=2)
    doc.add_paragraph("乙方定期对甲方排放的污水进行水质检测，检测结果作为污水处理费用结算的参考依据。")
    doc.add_heading("附件六：污水处理费用", level=2)
    table = doc.add_table(rows=2, cols=2)
    table.cell(0, 0).text, table.cell(0, 1).text = "项目", "单价"
    table.cell(1, 0).text, table.cell(1, 1).text = "污水处理费", "25元/吨"
    doc.save(TEST_DOCX)

contract_text = load_contract_docx(TEST_DOCX)
print(contract_text)

污水处理服务协议

6.3 水质超标处理费

甲方交付的污水水质超过本协议约定的接收标准时，乙方有权按照原污水处理单价的1.5倍收取超标处理费用。

6.4 水量结算

污水处理费用按照实际接收水量进行结算，甲方应按月支付污水处理费用。

5.1 水质检测

乙方定期对甲方排放的污水进行水质检测，检测结果作为污水处理费用结算的参考依据。

附件六：污水处理费用

[合同表格]
项目 | 单价
污水处理费 | 25元/吨


## 2. 先做离线验收（不花 API Token）

这是解析代码的验收，不是 LLM 审查质量测试。确认规则齐全、DOCX 内容齐全、表格位置正确。

In [32]:
assert isinstance(contract_text, str)
assert "1.5倍" in contract_text
assert "25元/吨" in contract_text
assert "按月支付" in contract_text
assert "定期对甲方排放的污水" in contract_text
assert contract_text.index("6.3 水质超标处理费") < contract_text.index("[合同表格]")
assert contract_text.index("5.1 水质检测") < contract_text.index("[合同表格]")
assert "\n" in contract_text and "/n" not in contract_text
print("离线验收通过：3 条规则、DOCX 段落+表格、原始顺序、换行符")

离线验收通过：3 条规则、DOCX 段落+表格、原始顺序、换行符


## 3. 定义结构化输出与单规则审查函数

下面复用今天 Windows 上已验证的 `ContractReviewResult` 字段和 `review_contract_rule()` 设计。

注意：`evidence` 可能被模型改写，因此暂时输出**原文精确匹配提示**，之后再做自动证据校验和重试机制。

In [33]:
from typing import Literal
from pydantic import BaseModel, Field


class ContractReviewResult(BaseModel):
    rule_id: str = Field(description="审查规则编号")
    status: Literal["符合要求", "存在问题", "无法判断"] = Field(description="只依据提供合同文本判断")
    evidence: str = Field(description="支持审查结论的合同原文")
    issue: str = Field(description="发现的问题及其具体原因")
    suggestion: str = Field(description="针对问题提出的合同修改建议")


def review_contract_rule(contract_text: str, rule: dict, structured_llm) -> ContractReviewResult:
    prompt = f"""
你是一名工业污水处理合同审查助手。
审查规则编号：{rule['rule_id']}
审查规则名称：{rule['title']}
审查问题：{rule['review_question']}
审查要求：{rule['expected_requirement']}
参考建议：{rule['suggestion']}
合同原文：{contract_text}
请严格按照以下要求审查：
1. 只依据提供的合同文本判断；不得编造合同内容。
2. status 只能是：符合要求、存在问题、无法判断。
3. evidence 必须引用合同原文；不能把规则建议写成合同原文。
4. 必须返回正确的 rule_id。
5. 不得自行编造具体价格、超标倍数或公司制度。
6. 新增商务建议必须明确标注为待双方确认。
7. 请返回完整结构化审查结果，字段不能缺失。
"""
    result = structured_llm.invoke(prompt)
    if result.rule_id != rule["rule_id"]:
        raise ValueError(f"规则编号不匹配：预期 {rule['rule_id']}，实际 {result.rule_id}")
    return result

## 4. 调用模型审查 3 条规则（会消耗 API Token）

先运行下一格初始化模型；然后再运行最后一格进行三次调用。如果你只想验证 DOCX 解析，可以暂时不运行本节。

In [34]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv(PROJECT_ROOT / ".env")
api_key = os.getenv("SILICONFLOW_API_KEY")
if not api_key:
    raise RuntimeError("未找到 SILICONFLOW_API_KEY；请检查项目根目录 .env")

llm = ChatOpenAI(
    api_key=api_key,
    base_url="https://api.siliconflow.cn/v1",
    model="deepseek-ai/DeepSeek-V3.2",
    temperature=0,
)
structured_llm = llm.with_structured_output(ContractReviewResult)
print("结构化审查模型初始化完成")

结构化审查模型初始化完成


In [35]:
results = []
for rule in rules:
    print(f"正在审查：{rule['rule_id']} {rule['title']}")
    result = review_contract_rule(contract_text, rule, structured_llm)
    results.append(result)
    print(result.model_dump())
    if result.evidence and result.evidence not in contract_text:
        print("⚠️ 证据原文未精确匹配：需要人工核对 evidence")

assert len(results) == 3
assert {r.rule_id for r in results} == {"WQ-001", "WQ-002", "WQ-003"}
print("完成 3 条规则审查；请人工核查每项结论和证据。")

正在审查：WQ-001 水质超标处理与价格协商机制
{'rule_id': 'WQ-001', 'status': '存在问题', 'evidence': '合同原文："6.3 水质超标处理费\n甲方交付的污水水质超过本协议约定的接收标准时，乙方有权按照原污水处理单价的1.5倍收取超标处理费用。"', 'issue': '合同约定了水质超标时的处理费用计算规则（原单价1.5倍），但未明确乙方是否有权拒收超标污水或选择接收。同时，合同未对严重超标导致原1.5倍计价机制不适用的情况，约定双方另行协商单价的机制。', 'suggestion': '建议补充：当污水水质严重超出约定接收标准、原超标收费机制不足以覆盖处理成本或技术风险时，双方可就该批次接收条件和处理单价另行书面协商，达成一致后执行；具体约定待双方确认。'}
⚠️ 证据原文未精确匹配：需要人工核对 evidence
正在审查：WQ-002 水量计量与结算机制
{'rule_id': 'WQ-002', 'status': '存在问题', 'evidence': '合同原文中仅提及“污水处理费用按照实际接收水量进行结算，甲方应按月支付污水处理费用”，未明确计量设备或依据、计量位置、抄表/确认周期以及计量争议处理方式。', 'issue': '合同未明确计量设备或依据、计量位置、抄表/确认周期以及计量争议处理方式，仅规定按实际接收水量结算和按月支付，不符合审查规则要求。', 'suggestion': '建议在合同中明确计量仪表与计量点、双方定期抄表及签字确认要求，并约定仪表故障或计量争议时的复核和结算依据；具体安排待双方确认。'}
⚠️ 证据原文未精确匹配：需要人工核对 evidence
正在审查：WQ-003 水质采样与检测机制
{'rule_id': 'WQ-003', 'status': '存在问题', 'evidence': '合同原文中仅提及“乙方定期对甲方排放的污水进行水质检测，检测结果作为污水处理费用结算的参考依据。”，未明确采样位置、检测频次、主要检测指标或检测依据、超标判定依据以及检测结果争议的复检机制。', 'issue': '合同未明确采样位置、检测频次、主要检测指标或检测依据、超标判定依据以及检测结果争议的复检机制。条款“定期检测”过于笼统，不符合审查规则要求。', 'suggestion

## 验收与后续

1. 第一格输出 WQ-001 / WQ-002 / WQ-003。
2. DOCX 提取结果按原序保留三个条款和 `[合同表格]`，`25元/吨` 可找到。
3. 离线验收全部通过。
4. 可选：模型返回三条 Pydantic 结构化审查结果。样例条款刻意不完整，合理的结果通常应指出计量确认、采样与复检以及严重超标协商机制的缺失；**具体结论仍须核对，不能仅凭 `assert` 判定审查质量**。

下一阶段：将 `load_contract_docx()` 和 `review_contract_rule()` 从 Notebook 迁移到 `src/contract_review.py`，补测试、失败重试和证据定位。

In [36]:
%pip install docx

Note: you may need to restart the kernel to use updated packages.


In [37]:
import docx

print(docx.__version__)
print(docx.__file__)

1.2.0
/opt/homebrew/Caskroom/miniforge/base/envs/wastewater-rag/lib/python3.12/site-packages/docx/__init__.py


In [38]:
from pathlib import Path
import sysconfig

site = Path(sysconfig.get_paths()["purelib"])

old_docx = site / "docx.py"
new_docx = site / "docx" / "__init__.py"

print("旧版文件存在：", old_docx.exists())
print("新版包存在：", new_docx.exists())

if old_docx.exists():
    old_docx.rename(site / "docx_legacy_backup.py")
    print("旧版文件已重命名备份")

旧版文件存在： False
新版包存在： True


In [39]:
%pip install --no-cache-dir --upgrade python-docx==1.2.0

Note: you may need to restart the kernel to use updated packages.


In [40]:
import docx

print(docx.__version__)
print(docx.__file__)

1.2.0
/opt/homebrew/Caskroom/miniforge/base/envs/wastewater-rag/lib/python3.12/site-packages/docx/__init__.py


In [54]:
def load_contract_docx(file_path: str | Path) -> str:

    doc = Document(file_path)

    contract_blocks = []

    for block in doc.iter_inner_content():

        if isinstance(block, Paragraph):

            text = block.text.strip()

            if text:
                contract_blocks.append(text)

        elif isinstance(block, Table):

            table_rows = []
            for row in block.rows:
                row_text = []
                for cell in row.cells:
                    row_text.append(cell.text)
                row_content = '|'.join(row_text)
                table_rows.append(row_content)
            table_content = '\n'.join(table_rows)
            contract_blocks.append(table_content)
    return '\n\n'.join(contract_blocks)
                
                
                


In [55]:
test_path = Path('../data/raw/contracts/test_contract_win_rebuild.docx')

In [56]:
print(test_path)

../data/raw/contracts/test_contract_win_rebuild.docx


In [57]:
contract_text = load_contract_docx(test_path)

In [58]:
print(contract_text)

污水处理服务协议

6.3 水质超标处理费

甲方交付的污水水质超过本协议约定的接收标准时，乙方有权按照原污水处理单价的1.5倍收取超标处理费用。

6.4 水量结算

污水处理费用按照实际接收水量进行结算，甲方应按月支付污水处理费用。

5.1 水质检测

乙方定期对甲方排放的污水进行水质检测，检测结果作为污水处理费用结算的参考依据。

附件六：污水处理费用

项目|单价
污水处理费|25元/吨


In [59]:
def load_contract_rules(file_path: str | Path) -> list[dict]:

    # 1. 打开 JSON 文件

    # 2. 用 json.load() 读取内容

    # 3. 返回读取结果
    with open(file_path,'r',encoding='utf-8') as f:
        context = json.load(f)
    return context

In [62]:
rule_path = Path('../data/rules/contract_rules_win_rebuild.json')
rules = load_contract_rules(
    rule_path
)

print(type(rules))
print(len(rules))
for rule in rules :
    print(rule)

<class 'list'>
3
{'rule_id': 'WQ-001', 'title': '水质超标处理与价格协商机制', 'category': '水质管理', 'risk_level': '高', 'review_question': '合同是否约定水质超标时的拒收、接收与费用处理机制？严重超标且原计价机制不适用时，是否保留双方另行协商单价的机制？', 'expected_requirement': '应明确污水超标时乙方的拒收或选择接收权、适用的超标处理费用计算规则，并对严重超标或既定倍数不适用的情形保留双方协商确定处理价格的约定。不得根据审查规则虚构现有合同条款。', 'suggestion': '建议补充：当污水水质明显超出约定接收标准、原超标收费机制不足以覆盖处理成本或技术风险时，双方可就该批次接收条件和处理单价另行书面协商，达成一致后执行；具体约定待双方确认。'}
{'rule_id': 'WQ-002', 'title': '水量计量与结算机制', 'category': '计量结算', 'risk_level': '高', 'review_question': '合同是否明确计量设备或依据、计量位置、抄表/确认周期以及计量争议处理方式？', 'expected_requirement': '应明确计量依据及计量位置、双方确认水量的时间或频率、异常计量或数据争议时的核实/复核方式。仅写按实际接收量结算或按月付款，不等于已经明确月度抄表确认机制。', 'suggestion': '建议明确计量仪表与计量点、双方定期抄表及签字确认要求，并约定仪表故障或计量争议时的复核和结算依据；具体安排待双方确认。'}
{'rule_id': 'WQ-003', 'title': '水质采样与检测机制', 'category': '水质管理', 'risk_level': '高', 'review_question': '合同是否明确采样位置、检测频次、主要检测指标或检测依据、超标判定依据以及检测结果争议的复检机制？', 'expected_requirement': '应明确采样点位、频次或触发条件、适用检测指标/方法或附件、超标判定的限值依据，并约定双方对检测结果有争议时的复检办法。笼统写定期检测不足以认定符合要求。', 'suggestion': '建议明确

In [80]:
def review_contract_rule(
    contract_text: str,
    rule: dict,
    structured_llm
) -> ContractReviewResult:

    prompt = f'''
    你是一名工业污水处理业务合同审查助手。

请严格依据提供的合同文本和审查规则进行审查。

【审查规则】
规则编号：{rule['rule_id']}
规则名称：{rule['title']}
审查问题：{rule['review_question']}
预期要求：{rule['expected_requirement']}
参考建议：{rule['suggestion']}

【合同原文】
{contract_text}

【审查要求】
1. 只能依据提供的合同原文进行判断，不得编造条款。
2. 如果合同已经满足审查要求，status返回"符合要求"。
3. 如果合同存在缺失、表述模糊或不满足审查要求，status返回"存在问题"。
4. 如果现有文本不足以作出判断，status返回"无法判断"。
5. evidence必须直接复制合同原文中一段连续的关键文字。不得改写、概括或拼接不同位置的内容。优先引用完整句子，不必同时引用章节标题。
6. issue应说明具体问题；如果符合要求，可以留空。当status为“符合要求”时，issue必须返回空字符串""，
不得在issue字段中描述合同符合要求的原因。
7. suggestion应提出具有可操作性的修改建议。
8. 不得自行编造价格、超标倍数及公司制度。
9. 涉及新增商务条件时，必须注明需双方协商确认。
10. 返回的rule_id必须与输入规则编号一致。

请返回完整的结构化审查结果。
    '''
    response = structured_llm.invoke(prompt)
    if response.rule_id != rule['rule_id']:
        raise ValueError('规则编码错误')
    return response
    
    # 1. 构建 prompt，包含规则信息和合同原文

    # 2. 调用 structured_llm.invoke(prompt)

    # 3. 校验返回的 rule_id 是否与输入规则一致

    # 4. 返回审查结果

In [70]:
result = review_contract_rule(contract_text,rules[0],structured_llm)

print(result)

rule_id='WQ-001' status='存在问题' evidence='甲方交付的污水水质超过本协议约定的接收标准时，乙方有权按照原污水处理单价的1.5倍收取超标处理费用。' issue='合同仅约定了水质超标时按固定倍数（1.5倍）收取处理费，但未明确乙方是否有权拒收或选择接收超标污水。同时，合同未对严重超标导致原计价机制（1.5倍）不适用或不足以覆盖处理成本的情形，约定双方另行协商处理价格的机制。' suggestion='建议在6.3条中补充以下内容：1. 明确乙方对超标污水的处置选择权，例如“乙方有权选择拒收或接收该批次超标污水。若选择接收，则按本条约定的超标处理费标准执行。” 2. 增加价格协商机制，例如“若污水水质严重超出约定标准，导致本条约定的超标处理费标准明显不足以覆盖乙方处理成本或技术风险时，双方应就该批次污水的接收条件及处理单价另行进行书面协商，协商一致后执行。具体约定需经双方书面确认。”'


In [71]:
print(result.model_dump())

{'rule_id': 'WQ-001', 'status': '存在问题', 'evidence': '甲方交付的污水水质超过本协议约定的接收标准时，乙方有权按照原污水处理单价的1.5倍收取超标处理费用。', 'issue': '合同仅约定了水质超标时按固定倍数（1.5倍）收取处理费，但未明确乙方是否有权拒收或选择接收超标污水。同时，合同未对严重超标导致原计价机制（1.5倍）不适用或不足以覆盖处理成本的情形，约定双方另行协商处理价格的机制。', 'suggestion': '建议在6.3条中补充以下内容：1. 明确乙方对超标污水的处置选择权，例如“乙方有权选择拒收或接收该批次超标污水。若选择接收，则按本条约定的超标处理费标准执行。” 2. 增加价格协商机制，例如“若污水水质严重超出约定标准，导致本条约定的超标处理费标准明显不足以覆盖乙方处理成本或技术风险时，双方应就该批次污水的接收条件及处理单价另行进行书面协商，协商一致后执行。具体约定需经双方书面确认。”'}


In [72]:
print(result.evidence in contract_text)

True


In [68]:
import re

clean_contract = re.sub(r"\s+", "", contract_text)
clean_evidence = re.sub(r"\s+", "", result.evidence)

print(clean_evidence in clean_contract)

True


In [73]:
def review_contract(
    contract_text: str,
    rules: list[dict],
    structured_llm
) -> list[ContractReviewResult]:

    results = []
    for rule in rules:
        response = review_contract_rule(contract_text,rule,structured_llm)
        results.append(response)
    return results

In [74]:
results = review_contract(contract_text, rules, structured_llm)

assert len(results) == 3

for result in results:
    print(result.rule_id, result.status)

WQ-001 存在问题
WQ-002 存在问题
WQ-003 存在问题


In [75]:
for result in results:
    print(f"\n规则编号：{result.rule_id}")
    print(f"审查结论：{result.status}")
    print(f"引用证据：{result.evidence}")
    print(f"发现问题：{result.issue}")
    print(f"修改建议：{result.suggestion}")
    print(f"证据原文匹配：{result.evidence in contract_text}")
    print("-" * 50)


规则编号：WQ-001
审查结论：存在问题
引用证据：甲方交付的污水水质超过本协议约定的接收标准时，乙方有权按照原污水处理单价的1.5倍收取超标处理费用。
发现问题：合同仅约定了超标时按固定倍数（1.5倍）收取处理费，但未明确乙方是否有权拒收或选择接收超标污水。同时，合同未对‘严重超标’或‘原计价机制不适用’（例如，1.5倍单价不足以覆盖处理成本或技术风险）的情形作出约定，缺乏双方另行协商处理单价的机制。
修改建议：建议在6.3条中补充以下内容：
1. 明确乙方权利：当污水水质超过约定接收标准时，乙方有权选择拒收或接收。
2. 增加协商机制：当污水水质严重超标，或原1.5倍单价明显不足以覆盖乙方因此增加的处理成本及技术风险时，双方应就该批次污水的接收条件及处理单价进行另行书面协商，协商一致后执行。具体协商标准及流程需双方另行确认。
证据原文匹配：True
--------------------------------------------------

规则编号：WQ-002
审查结论：存在问题
引用证据：污水处理费用按照实际接收水量进行结算，甲方应按月支付污水处理费用。
发现问题：合同未明确计量设备或依据、计量位置、抄表/确认周期以及计量争议处理方式。条款仅提及“按实际接收量结算”和“按月支付”，未建立具体的月度抄表确认机制，也未约定仪表故障或数据争议时的处理办法。
修改建议：建议增加专门条款，明确约定：1. 计量仪表的具体类型、安装位置（计量点）；2. 双方每月共同抄表并签字确认水量的具体时间和程序；3. 当出现仪表故障、数据异常或双方对计量结果有争议时的复核方式（如启用备用表、共同委托第三方检测等）及临时结算依据。具体安排需双方协商确认。
证据原文匹配：True
--------------------------------------------------

规则编号：WQ-003
审查结论：存在问题
引用证据：乙方定期对甲方排放的污水进行水质检测，检测结果作为污水处理费用结算的参考依据。
发现问题：合同未明确采样位置、检测频次、主要检测指标或检测依据、超标判定依据以及检测结果争议的复检机制。仅约定“定期检测”，表述模糊，不足以满足审查规则要求。
修改建议：建议在合同或附件中明确以下内容，具体细节需双方协商确认：1. 采样点位（如：甲方排放口、乙方

In [76]:
good_contract = """
6.3 水质超标处理

当甲方交付的污水超过约定接收标准时，
乙方有权选择拒收或接收。
若乙方同意接收，原则上按合同约定的超标处理费标准收费。
对于严重超标或原收费机制不足以覆盖处理成本的情形，
双方另行书面协商接收条件及处理价格，
协商未达成一致前乙方有权拒收。

6.4 水量计量

污水接收水量以双方确认的接收点计量仪表读数为准。
双方每月最后一个工作日共同抄表并签字确认。
如仪表故障或计量结果存在争议，
双方应共同核查仪表记录并根据确认的备用计量数据结算；
仍有争议的，由双方共同委托第三方核验。

6.5 水质采样与检测

乙方在双方约定的污水接收点进行采样，
每批次接收时检测COD、氨氮、总磷及pH。
检测方法、各指标接收限值及超标判定依据
按照合同附件《污水接收水质标准》执行。
如双方对检测结果存在争议，
应封存留样并共同委托有资质的第三方复检，
复检结果作为争议处理依据。
"""

In [77]:
good_results = review_contract(good_contract, rules, structured_llm)

for result in good_results:
    print(result.rule_id, result.status)

WQ-001 符合要求
WQ-002 符合要求
WQ-003 符合要求


In [78]:
# 从完整合同中删除月度共同抄表确认条款
mixed_contract = good_contract.replace(
    "双方每月最后一个工作日共同抄表并签字确认。",
    ""
)

assert mixed_contract != good_contract

mixed_results = review_contract(
    mixed_contract,
    rules,
    structured_llm
)

for result in mixed_results:
    print(result.rule_id, result.status)
    print(result.issue)

WQ-001 符合要求

WQ-002 存在问题
合同未明确双方定期抄表及签字确认水量的具体时间或频率（如月度抄表确认机制）。
WQ-003 存在问题
合同条款未明确采样频次或触发条件。条款中“每批次接收时检测”的表述，对于连续接收或非批次接收模式下的检测频次约定不够明确，可能引发争议。


In [81]:
batch_contract = (
    "【业务场景】本合同仅适用于罐车分批接收污水，"
    "不涉及管道连续接收。\n\n"
    + mixed_contract
)

result = review_contract_rule(
    batch_contract,
    rules[2],
    structured_llm
)

print(result.status)
print(result.issue)

存在问题
合同未明确采样位置的具体约定方式，仅提及“双方约定的污水接收点”，但未说明该约定是否已明确或需后续确认。
